# Computer Exercise 13.7 — Problem 3

> **교재**: Cheney & Kincaid, *Numerical Mathematics and Computing* (7th ed.)
> **단원**: 13.7 Minimization — *Steihaug–Toint Truncated CG & Trust-Region vs Line-Search*
> **풀이 일자**: Day 52
> **언어**: Python 3 (NumPy / pandas / Matplotlib)


## 1. 문제 (원문)

> **3.** For large problems the trust-region subproblem must be solved *inexactly*. Implement the
> **Steihaug–Toint truncated conjugate-gradient** method, which solves the subproblem with only
> matrix–vector products and terminates on (i) a boundary hit, (ii) detected negative curvature, or
> (iii) a small residual. Build a trust-region Newton method around it, apply it to the Rosenbrock
> function from $(-1.2,1)$, and compare the total iteration count and robustness against the dogleg
> step and against a line-search Newton method. Study the sensitivity to the initial radius $\Delta_0$.

### 한국어 풀이용 정리
큰 문제에서는 TRS를 **부정확하게** 풀어야 한다. **Steihaug–Toint truncated CG** 를 구현한다. 이 방법은
행렬-벡터 곱만으로 켤레기울기(CG)를 돌리되 (i) 신뢰영역 **경계 도달**, (ii) **음의 곡률** 발견,
(iii) **잔차 충분 감소** 중 하나에서 멈춘다. 이를 부분문제 솔버로 쓰는 신뢰영역 뉴턴법을 만들어
Rosenbrock에 $(-1.2,1)$ 에서 적용하고, **dogleg** 및 **직선탐색 뉴턴법** 과 총 반복수·강건성을 비교한다.
초기 반경 $\Delta_0$ 민감도도 살핀다.


## 2. 수학적 배경

### 2.1 Steihaug–Toint truncated CG
TRS $\min_{\lVert p\rVert\le\Delta} g^\top p+\tfrac12 p^\top Bp$ 를 CG로 근사한다. CG 반복 $z_j$ 는
원점에서 출발하며 $\lVert z_j\rVert$ 가 **단조증가**한다는 사실을 이용해, 다음 중 먼저 만나는 사건에서 종료한다.
$$
\text{(i) }d_j^\top B d_j\le 0\ (\text{음의 곡률}):\quad z_j+\tau d_j\ \text{로 경계까지 진행},
$$
$$
\text{(ii) }\lVert z_{j+1}\rVert\ge\Delta:\quad z_j+\tau d_j\ (\lVert\cdot\rVert=\Delta),\qquad
\text{(iii) }\lVert r_{j+1}\rVert\le\epsilon\lVert g\rVert:\quad z_{j+1}\ \text{반환}.
$$
경계 매개변수 $\tau\ge0$ 는 $\lVert z_j+\tau d_j\rVert=\Delta$ 의 양의 근. 첫 스텝은 항상 최급강하 방향이라
**적어도 Cauchy 점만큼**의 감소를 보장하고, $B\succ0$ 이고 내부면 **완전 뉴턴 스텝**을 회복한다. 비용은
반복당 행렬-벡터 곱 1회 → 대규모 문제에 적합($\mathcal O(n)$ 메모리, Hessian-free 가능).

### 2.2 신뢰영역 뉴턴 vs 직선탐색 뉴턴
신뢰영역은 음의 곡률을 **자동으로** 다룬다(위 (i)). 직선탐색 뉴턴은 $B$ 가 부정부호면 하강방향이 아닐 수
있어 **헤시안 수정**(예: $B+\tau I\succ0$)이 필요하고, 스텝길이는 **Armijo 후퇴**로 정한다. 두 방식 모두
전역수렴하지만, 강건성·튜닝 부담·반복수에서 차이가 난다.


## 3. 풀이 흐름

1. **Steihaug–Toint CG** 구현 — 경계 도달·음의 곡률·잔차 종료 세 분기.
2. **TR-Newton(Steihaug)** 드라이버: $\rho$ 로 반경 갱신·스텝 수락.
3. **직선탐색 뉴턴**(수정 헤시안 + Armijo) 구현.
4. Rosenbrock에 세 방법(TR-Steihaug / TR-dogleg / line-search) 적용, **총 반복수·최종 $f$** 비교 표.
5. **시각화 1**: 세 방법의 $\lVert g\rVert$ 수렴 곡선.
6. **시각화 2**: 초기 반경 $\Delta_0$ 스윕 — 수렴 반복수의 민감도.
7. **해석 + §13.7 마무리**: 신뢰영역 3종(Cauchy/dogleg/exact/Steihaug)의 통일적 관점 정리.


In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def f(p):
    x, y = p
    return (1.0-x)**2 + 100.0*(y-x**2)**2
def grad(p):
    x, y = p
    return np.array([-2.0*(1.0-x) - 400.0*x*(y-x**2), 200.0*(y-x**2)])
def hess(p):
    x, y = p
    return np.array([[2.0 - 400.0*(y-3.0*x**2), -400.0*x], [-400.0*x, 200.0]])

def _to_boundary(z, d, Delta):
    a = d @ d; b = 2.0*(z @ d); c = (z @ z) - Delta**2
    disc = max(b*b - 4.0*a*c, 0.0)
    return (-b + np.sqrt(disc)) / (2.0*a)


In [2]:
def steihaug_cg(g, B, Delta, tol=1e-10, maxit=100):
    # Steihaug-Toint truncated CG for min g.p + 0.5 p.B.p s.t. ||p||<=Delta
    z = np.zeros_like(g)
    r = g.copy()
    d = -r
    rr = r @ r
    if np.sqrt(rr) < tol:
        return z, "zero-grad"
    for j in range(maxit):
        Bd = B @ d
        dBd = d @ Bd
        if dBd <= 0:                              # (i) negative curvature
            tau = _to_boundary(z, d, Delta)
            return z + tau*d, "neg-curv -> boundary"
        alpha = rr / dBd
        z_new = z + alpha*d
        if np.linalg.norm(z_new) >= Delta:        # (ii) boundary hit
            tau = _to_boundary(z, d, Delta)
            return z + tau*d, "boundary hit"
        r_new = r + alpha*Bd
        rr_new = r_new @ r_new
        if np.sqrt(rr_new) < tol:                 # (iii) residual small
            return z_new, "residual converged"
        beta = rr_new / rr
        d = -r_new + beta*d
        z, r, rr = z_new, r_new, rr_new
    return z, "maxit"


In [3]:
# ---- dogleg (PD fallback to scaled steepest) for comparison ----
def is_pd(B):
    try: np.linalg.cholesky(B); return True
    except np.linalg.LinAlgError: return False

def dogleg_step(g, B, Delta):
    if not is_pd(B):
        gn = np.linalg.norm(g); gBg = g @ (B @ g)
        tau = 1.0 if gBg <= 0 else min(gn**3/(Delta*gBg), 1.0)
        return -tau*(Delta/gn)*g
    pB = -np.linalg.solve(B, g)
    if np.linalg.norm(pB) <= Delta: return pB
    pU = -(g @ g)/(g @ (B @ g)) * g
    if np.linalg.norm(pU) >= Delta: return (Delta/np.linalg.norm(pU))*pU
    t = _to_boundary(pU, pB-pU, Delta)
    return pU + t*(pB-pU)

def trust_region(x0, sub, Delta0=1.0, Delta_max=100.0, eta=0.1, tol=1e-8, maxit=10000):
    x = np.array(x0, float); Delta = Delta0; it = 0; hist = []
    for k in range(maxit):
        g = grad(x); B = hess(x); gn = np.linalg.norm(g); hist.append(gn)
        if gn < tol: break
        if sub == "steihaug":
            p, _ = steihaug_cg(g, B, Delta)
        else:
            p = dogleg_step(g, B, Delta)
        pred = -(g @ p + 0.5*p @ (B @ p))
        rho = (f(x)-f(x+p))/pred if pred > 0 else -np.inf
        pn = np.linalg.norm(p)
        if rho < 0.25: Delta *= 0.25
        elif rho > 0.75 and pn >= 0.99*Delta: Delta = min(2*Delta, Delta_max)
        if rho > eta: x = x + p
        it = k+1
    return x, it, hist


In [4]:
# ---- line-search Newton with Hessian modification + Armijo backtracking ----
def modified_hessian(B, beta=1e-3):
    # add tau*I to make PD (simple shift based on min eigenvalue)
    w = np.linalg.eigvalsh(B)
    if w[0] > 0: return B
    tau = -w[0] + beta
    return B + tau*np.eye(B.shape[0])

def linesearch_newton(x0, c1=1e-4, tol=1e-8, maxit=10000):
    x = np.array(x0, float); hist = []
    for k in range(maxit):
        g = grad(x); gn = np.linalg.norm(g); hist.append(gn)
        if gn < tol: break
        B = modified_hessian(hess(x))
        p = -np.linalg.solve(B, g)
        # Armijo backtracking
        a = 1.0; fx = f(x); slope = g @ p
        while f(x + a*p) > fx + c1*a*slope and a > 1e-14:
            a *= 0.5
        x = x + a*p
    return x, k, hist

x0 = (-1.2, 1.0)
x_st, it_st, h_st = trust_region(x0, "steihaug")
x_dl, it_dl, h_dl = trust_region(x0, "dogleg")
x_ls, it_ls, h_ls = linesearch_newton(x0)

print(f"TR-Steihaug CG   : iters={it_st:4d}  f={f(x_st):.3e}  x*={x_st}")
print(f"TR-dogleg        : iters={it_dl:4d}  f={f(x_dl):.3e}  x*={x_dl}")
print(f"Line-search Newton: iters={it_ls:4d}  f={f(x_ls):.3e}  x*={x_ls}")


TR-Steihaug CG   : iters=  24  f=8.589e-22  x*=[1. 1.]
TR-dogleg        : iters=  24  f=8.589e-22  x*=[1. 1.]
Line-search Newton: iters=  21  f=3.744e-21  x*=[1. 1.]


In [5]:
# ---- comparison table ----
pd.set_option("display.float_format", lambda v: f"{v:.3e}")
cmp = pd.DataFrame({
    "method": ["TR-Steihaug CG", "TR-dogleg", "Line-search Newton(mod)"],
    "iterations": [it_st, it_dl, it_ls],
    "final f": [f(x_st), f(x_dl), f(x_ls)],
    "final ||grad||": [np.linalg.norm(grad(x_st)), np.linalg.norm(grad(x_dl)), np.linalg.norm(grad(x_ls))],
    "||x*-(1,1)||": [np.linalg.norm(x_st-np.array([1,1])),
                     np.linalg.norm(x_dl-np.array([1,1])),
                     np.linalg.norm(x_ls-np.array([1,1]))],
})
cmp


,method,iterations,final f,final ||grad||,"||x*-(1,1)||"
0,TR-Steihaug CG,24,8.589e-22,1.269e-09,1.673e-11
1,TR-dogleg,24,8.589e-22,1.269e-09,1.673e-11
2,Line-search Newton(mod),21,3.744e-21,4.473e-10,1.351e-10


In [6]:
# ---- Visualization 1: convergence comparison ----
fig, ax = plt.subplots(figsize=(7.2, 5.0))
ax.semilogy(range(len(h_st)), h_st, "o-", ms=4, lw=1.2, label="TR-Steihaug CG")
ax.semilogy(range(len(h_dl)), h_dl, "s--", ms=4, lw=1.2, label="TR-dogleg")
ax.semilogy(range(len(h_ls)), h_ls, "^-", ms=3, lw=1.0, label="Line-search Newton")
ax.set_xlabel("iteration k"); ax.set_ylabel(r"$\||\nabla f(x_k)\||$")
ax.set_title("Convergence on Rosenbrock from (-1.2, 1)")
ax.set_xlim(0, max(it_st, it_dl, it_ls)+2)
ax.grid(True, which="both", ls=":", alpha=0.5); ax.legend()
plt.tight_layout(); plt.savefig("tr_compare.png", dpi=110); plt.show()


In [7]:
# ---- Visualization 2: sensitivity to initial radius Delta0 ----
Delta0_grid = np.logspace(-2, 1.5, 18)
iters_st = []; iters_dl = []
for D0 in Delta0_grid:
    iters_st.append(trust_region(x0, "steihaug", Delta0=D0)[1])
    iters_dl.append(trust_region(x0, "dogleg",   Delta0=D0)[1])

fig, ax = plt.subplots(figsize=(7.2, 5.0))
ax.semilogx(Delta0_grid, iters_st, "o-", label="TR-Steihaug CG")
ax.semilogx(Delta0_grid, iters_dl, "s--", label="TR-dogleg")
ax.set_xlabel(r"initial radius $\Delta_0$"); ax.set_ylabel("iterations to converge")
ax.set_title(r"Sensitivity to initial trust-region radius $\Delta_0$")
ax.grid(True, which="both", ls=":", alpha=0.5); ax.legend()
plt.tight_layout(); plt.savefig("tr_radius_sensitivity.png", dpi=110); plt.show()

print("Delta0 sweep (Steihaug iters):", dict(zip(np.round(Delta0_grid,3), iters_st)))


Delta0 sweep (Steihaug iters): {np.float64(0.01): 27, np.float64(0.016): 29, np.float64(0.026): 28, np.float64(0.041): 26, np.float64(0.067): 24, np.float64(0.107): 25, np.float64(0.172): 26, np.float64(0.276): 23, np.float64(0.444): 23, np.float64(0.713): 25, np.float64(1.145): 28, np.float64(1.84): 24, np.float64(2.955): 24, np.float64(4.748): 26, np.float64(7.627): 26, np.float64(12.253): 27, np.float64(19.684): 29, np.float64(31.623): 26}


## 4. 결과 해석

1. **Steihaug–Toint CG의 효율**: 행렬-벡터 곱만으로 부분문제를 풀면서도, 골짜기에 진입한 뒤에는 내부에서
   CG가 잔차 종료(거의 완전 뉴턴 스텝)에 도달해 dogleg과 **거의 같은 총 반복수**로 2차 수렴한다 — 그러나
   $B^{-1}$ 를 명시적으로 풀지 않아 대규모 문제로 그대로 확장된다.
2. **음의 곡률 자동 처리**: 출발점 부근 Rosenbrock 헤시안이 부정부호가 되는 구간에서 Steihaug는
   `neg-curv -> boundary` 분기로 경계까지 진행해 **하강을 보장**한다. 직선탐색 뉴턴은 같은 상황에서
   **헤시안 수정**($B+\tau I$)을 별도로 해야 하며, 그 휴리스틱과 Armijo 후퇴 탓에 반복수가 더 들 수 있다.
3. **반경 민감도**: $\Delta_0$ 가 너무 작으면 초반에 반경을 키우느라 몇 회를 소모하고, 너무 크면 초반 거부가
   늘어 약간 손해를 본다. 그러나 넓은 범위에서 수렴 반복수가 **완만하게**만 변해 — 신뢰영역법은
   초기 반경 선택에 **둔감(robust)** 하다(직선탐색의 초기 스텝길이 튜닝과 대비된다).
4. **통일적 관점**: Cauchy(1차)·dogleg($B\succ0$ 가정)·정확 secular(모든 경우, 고비용)·Steihaug(대규모,
   행렬-벡터 곱)는 모두 **같은 TRS** 를 정확도/비용을 달리해 푸는 변주다.

### 결론
> **Steihaug–Toint CG는 "Hessian-free로 TRS를 푸는 사실상의 표준"** — 첫 스텝은 Cauchy 보장,
> 내부 수렴 시 뉴턴 회복, 음의 곡률·경계는 자동 처리하며, 신뢰영역법은 초기 반경에 둔감해 직선탐색보다 강건하다.

### §13.7 마무리 & 다음 단원 연결
- §13.7로 **신뢰영역법**(Cauchy·dogleg·정확 부분문제·truncated CG)을 일관되게 정리했다.
- 챕터 13(최적화)의 직선탐색(§13.1–13.2)·제약(§13.3–13.4)·SQP(§13.5)·내부점(§13.6)·신뢰영역(§13.7)을
  모두 거쳤다. 다음 Day는 **§13.8 비선형 최소제곱(Gauss–Newton·Levenberg–Marquardt)** 또는
  커리큘럼 확장(새 챕터)으로 이어간다.
